## Data Cleaning for corn data

### Corn and ethanol price data

In [44]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

df_corn_price = pd.read_excel("../data/raw-data/corn_ethanol_prices.xlsx")

df_corn_price.columns

Index(['Year', 'Month', 'Corn', 'Ethanol',
       'Blender_cost_of_ethanol_with_credit', 'Gasoline',
       'Corn_cost_per_gallon_of_ethanol', 'Cost_of_ethanol_geg'],
      dtype='object')

In [45]:
# List out fiscal year quarters

quarter_1 = ['Sep', 'Oct', 'Nov']
quarter_2 = ['Dec', 'Jan', 'Feb']
quarter_3 = ['Mar', 'Apr', 'May']
quarter_4 = ['Jun', 'Jul', 'Aug']

df_corn_price["fiscal_quarter"] = None

# Loop through dataframe and match each month to their respective quarter

for i in range(len(df_corn_price)):
    if df_corn_price["Month"][i] in quarter_1:
        df_corn_price.loc[i, "fiscal_quarter"] = "Q1"

    elif df_corn_price["Month"][i] in quarter_2:
        df_corn_price.loc[i, "fiscal_quarter"] = "Q2"

    elif df_corn_price["Month"][i] in quarter_3:
        df_corn_price.loc[i, "fiscal_quarter"] = "Q3"

    elif df_corn_price["Month"][i] in quarter_4:
        df_corn_price.loc[i, "fiscal_quarter"] = "Q4"

# Put year and fiscal quarter columns together into 1 column
df_corn_price["year_quarter"] = df_corn_price["Year"].astype(str) + '_' + df_corn_price["fiscal_quarter"]

In [46]:
df_corn_price_quarter = df_corn_price.groupby("year_quarter").agg({'Corn': 'mean', 'Ethanol': 'mean'}).reset_index()

In [47]:
df_corn_price_quarter = df_corn_price_quarter.rename(columns={"Corn": "corn_price", "Ethanol": "ethanol_price"})



### Corn production split data

In [48]:
df_corn_ethanol_share = pd.read_excel("../data/raw-data/corn_ethanol_share.xlsx")

In [49]:
df_corn_ethanol_share["year_quarter"] = df_corn_ethanol_share["Marketing_year_1"].astype(str) + '_' + df_corn_ethanol_share["Marketing_year_quarter"].str[:2]

In [50]:
# Create percentage share for each category, alcohol is already made

df_corn_ethanol_share["feed_share"] = (df_corn_ethanol_share["Feed_use"] / df_corn_ethanol_share["Total_use"]) *100

df_corn_ethanol_share["food_seed_indust_share"] = (df_corn_ethanol_share["Food_seed_and_industrial_use"] / df_corn_ethanol_share["Total_use"]) *100

df_corn_ethanol_share["exports_share"] = (df_corn_ethanol_share["Exports"] / df_corn_ethanol_share["Total_use"]) * 100

In [51]:
# Create quarterly only data
df_corn_alc_shr_quarterly = df_corn_ethanol_share[df_corn_ethanol_share["year_quarter"].str[5:] != "MY"].reset_index().drop('index', axis=1)

# Create yearly only data
df_corn_alc_shr_yearly = df_corn_ethanol_share[df_corn_ethanol_share["year_quarter"].str[5:] == "MY"].reset_index().drop('index', axis=1)

In [52]:
df_corn_alc_shr_quarterly

,Marketing_year_1,Marketing_year_quarter,Total_supply,Fuel_alcohol_use,Food_seed_and_industrial_use,Feed_use,Exports,Total_use,Fuel_alcohol_share_of_total_use,year_quarter,feed_share,food_seed_indust_share,exports_share
0,1986,Q1 Sep-Nov,12265.997,67.943,226.530,1347.813,318.212,1960.498,3.465599,1986_Q1,68.748502,11.554717,16.231182
1,1986,Q2 Dec-Feb,10305.666,71.514,209.977,1463.150,312.831,2057.473,3.475817,1986_Q2,71.113934,10.205577,15.204622
2,1986,Q3 Mar-May,8248.642,77.433,255.180,1087.691,496.097,1916.401,4.040543,1986_Q3,56.756963,13.315585,25.886910
3,1986,Q4 Jun-Aug,6332.671,73.102,251.732,760.813,365.331,1450.978,5.038119,1986_Q4,52.434496,17.349126,25.178259
4,1987,Q1 Sep-Nov,12013.555,67.245,228.888,1550.856,395.561,2242.550,2.998595,1987_Q1,69.155916,10.206595,17.638893
...,...,...,...,...,...,...,...,...,...,...,...,...,...
153,2024,Q2 Dec-Feb,12080.067,1369.120,319.611,1548.740,695.159,3932.630,34.814361,2024_Q2,39.381788,8.127157,17.676695
154,2024,Q3 Mar-May,8153.242,1320.556,377.119,929.519,883.154,3510.348,37.618948,2024_Q3,26.479397,10.743066,25.158588
155,2024,Q4 Jun-Aug,4648.383,1364.157,343.500,626.684,762.756,3097.097,44.046312,2024_Q4,20.234562,11.091031,24.628095
156,2025,Q1 Sep-Nov,18578.146,1374.852,322.483,2753.603,821.383,5272.321,26.076789,2025_Q1,52.227529,6.116528,15.579154


## Data cleaning for Soybean, poultry, and pork data

In [53]:
# Helper function to change FRED data into same format as corn price data

def to_marketing_quarter(df, value_col, new_col_name):
    df['date'] = pd.to_datetime(df['observation_date'])
    
    def get_mkt_quarter(date):
        month = date.month
        year = date.year
        
        if month in [9, 10, 11]:      # MYQ1
            return f"{year}_Q1"
        elif month in [12, 1, 2]:     # MYQ2
            marketing_year = year if month == 12 else year - 1
            return f"{marketing_year}_Q2"
        elif month in [3, 4, 5]:      # MYQ3
            return f"{year - 1}_Q3"
        else:                          # MYQ4 Jun, Jul, Aug
            return f"{year - 1}_Q4"
    
    df['year_quarter'] = df['date'].apply(get_mkt_quarter)
    df_grouped = df.groupby('year_quarter')[value_col].mean().reset_index()
    df_grouped = df_grouped.rename(columns={value_col: new_col_name})
    
    return df_grouped

### Soybean price data

In [54]:
# Import soybean data

df_soybean_price = pd.read_csv("../data/raw-data/soybean_prices.csv")

df_soybean_price.head()

,observation_date,WPU01830131
0,1947-01-01,51.8
1,1947-02-01,53.5
2,1947-03-01,65.0
3,1947-04-01,62.7
4,1947-05-01,48.7


In [55]:
df_soybean_clean_price = to_marketing_quarter(df_soybean_price, "WPU01830131", "soybean_price")

df_soybean_clean_price.head()

,year_quarter,soybean_price
0,1946_Q2,52.650000
1,1946_Q3,58.800000
2,1946_Q4,53.466667
3,1947_Q1,56.866667
4,1947_Q2,65.066667


### Poultry price data 

In [56]:
df_poultry_price = pd.read_csv("../data/raw-data/poultry_prices.csv")

df_poultry_price.head()

,observation_date,WPS014
0,1967-01-01,56.6
1,1967-02-01,58.5
2,1967-03-01,56.5
3,1967-04-01,57.3
4,1967-05-01,53.6


In [57]:
df_poultry_clean_price = to_marketing_quarter(df_poultry_price, "WPS014", "poultry_price")

df_poultry_clean_price.head()

,year_quarter,poultry_price
0,1966_Q2,57.550000
1,1966_Q3,55.800000
2,1966_Q4,51.066667
3,1967_Q1,46.666667
4,1967_Q2,50.433333


### Pork price data

In [58]:
df_pork_price = pd.read_csv("../data/raw-data/pork_prices.csv")

df_pork_price.head()

,observation_date,WPS022104
0,1974-01-01,67.0
1,1974-02-01,67.4
2,1974-03-01,64.6
3,1974-04-01,61.9
4,1974-05-01,56.2


In [59]:
df_pork_clean_price = to_marketing_quarter(df_pork_price, "WPS022104", "pork_price")

df_pork_clean_price.head()

,year_quarter,pork_price
0,1973_Q2,67.200000
1,1973_Q3,60.900000
2,1973_Q4,60.533333
3,1974_Q1,68.133333
4,1974_Q2,71.433333


## Joining price data

In [60]:
from functools import reduce

dfs = [df_corn_price_quarter, df_poultry_clean_price, df_soybean_clean_price, df_pork_clean_price]
df_merged = reduce(lambda left, right: pd.merge(left, right, on='year_quarter', how='inner'), dfs)

In [61]:
df_merged = df_merged[df_merged['year_quarter'] >= '2000_Q1']
df_merged = df_merged[df_merged['year_quarter'] <= '2015_Q4']

base = df_merged[df_merged['year_quarter'].str.startswith('2005')].mean(numeric_only=True)

# Index all columns a 2005 baseline
df_merged['corn_idx'] = (df_merged['corn_price'] / base['corn_price']) * 100
df_merged['poultry_idx'] = (df_merged['poultry_price'] / base['poultry_price']) * 100
df_merged['soybean_idx'] = (df_merged['soybean_price'] / base['soybean_price']) * 100
df_merged['pork_idx'] = (df_merged['pork_price'] / base['pork_price']) * 100
df_merged['ethanol_idx'] = (df_merged['ethanol_price'] / base['ethanol_price']) * 100

In [62]:
OUT_DIR = os.path.expanduser("../data/processed-data")
os.makedirs(OUT_DIR, exist_ok=True)

df_price_processed = df_merged.copy()
df_price_processed.to_csv(os.path.join(OUT_DIR, "food_prices.csv"), index=False)

df_corn_pct = df_corn_alc_shr_quarterly.copy()
df_corn_pct.to_csv(os.path.join(OUT_DIR, "corn_pct_usage.csv"), index=False)